# 🚁 Notebook 06 — Derivative Control

### The shock absorber: anticipating the future

So far our controller looks at the **present** (P: the error now) and the **past** (I: the error's
running total). Neither one looks **ahead**. That's why a well-tuned PI drone *still* overshoots: it
keeps pushing up even as it's rushing toward the target, and only realizes it's overshot after the
fact.

The **D** in PID — **Derivative** — fixes this. It watches **how fast the error is changing** and
pushes *against* rapid motion, like a shock absorber or a careful driver easing off the gas *before*
reaching a stop sign. But D has a famous weakness: it **amplifies sensor noise**. We'll see exactly
why, and how to fix it.

---

## 🎯 Learning objectives
- Understand the derivative as the **rate of change (slope)** of the signal.
- Build a **PD controller** and watch D **damp** overshoot and oscillation.
- See *why* D **amplifies noise**, with your own eyes.
- Tame D with a **low-pass filter** and **derivative-on-measurement**.

## 🧩 Prerequisites
Notebooks 01–05 (especially P and I; overshoot; the reusable `simulate()` engine).

## ⏱️ Estimated time
About **50–65 minutes**.

## 🧠 Concepts you know so far
P control · I control · steady-state error · windup / anti-windup · the `simulate()` engine

## 🔜 Concepts you'll learn here
Derivative of error · slope / rate of change · damping & prediction · derivative gain (Kd) · noise amplification · low-pass filtering · derivative-on-measurement


### 🔁 Quick recap
P pushes proportional to the error *now*; I accumulates the error over *time*. Both can push the
drone into an overshoot. D looks at **how fast** things are changing and brakes accordingly. Run
setup, then bring in our reusable engine.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


### 🧰 Reusable simulator (same engine from Notebook 05)

In [ ]:
from collections import deque

def simulate(controller, target=10.0, mass=1.0, g=9.8, start_height=0.0,
             dt=0.02, total_time=12.0, max_thrust=30.0, min_thrust=0.0,
             noise_std=0.0, wind=0.0, delay_steps=0, seed=0):
    """Run the 1D drone with ANY controller and return recorded signals as arrays.

    `controller` is a callable: controller(target, measured_altitude, dt) -> desired_thrust.
    It may hold internal state (integral, previous error, etc.) and may expose a dict
    `controller.last_terms = {'p':..,'i':..,'d':..}` which we record for the dashboards.

    Physical extras (all optional, default off):
      noise_std  : Gaussian sensor noise standard deviation (metres)
      wind       : constant extra force in newtons (+ up, - down)
      delay_steps: how many steps the sensor reading lags behind reality
      max/min_thrust : actuator saturation limits (newtons)
    """
    rng = np.random.default_rng(seed)
    x, v, t = start_height, 0.0, 0.0
    buf = deque([start_height] * (delay_steps + 1), maxlen=delay_steps + 1)  # sensor delay line

    keys = ("t", "x", "v", "a", "measured", "thrust", "error", "p", "i", "d")
    hist = {k: [] for k in keys}

    for _ in range(int(total_time / dt)):
        buf.append(x)                                    # newest true altitude enters the line
        delayed = buf[0]                                 # controller sees the OLD reading
        measured = delayed + (rng.normal(0, noise_std) if noise_std > 0 else 0.0)
        error = target - measured

        thrust = controller(target, measured, dt)        # <-- the controller decides
        thrust = min(max(thrust, min_thrust), max_thrust)  # actuator can't exceed limits

        net_force = thrust - mass * g + wind             # sum of forces (up +, down -)
        a = net_force / mass                             # Newton's second law

        terms = getattr(controller, "last_terms", {"p": 0.0, "i": 0.0, "d": 0.0})
        for k, val in zip(keys, (t, x, v, a, measured, thrust, error,
                                 terms.get("p", 0.0), terms.get("i", 0.0), terms.get("d", 0.0))):
            hist[k].append(val)

        v = v + a * dt                                   # Euler integrate
        x = x + v * dt
        if x < 0:                                        # ground
            x, v = 0.0, 0.0
        t = t + dt

    return {k: np.array(val) for k, val in hist.items()}


## 1. The derivative idea (slope intuition)

"Derivative" is a fancy word for a simple thing: **how fast is a number changing?** — the **slope**
of its graph.

- Altitude rising quickly → steep upward slope → large positive rate of change.
- Altitude barely moving → flat → near-zero rate of change.

We estimate it the obvious way: *(new value − old value) / time between them*:

$$ \text{rate of change} \approx \frac{\text{measured}_{now} - \text{measured}_{before}}{dt} $$

Now the clever bit. If the drone is racing **upward** toward the target, that's a big *upward* rate
of change — a warning that we're about to overshoot. So D should **push down** to slow the approach.
That's why the D term gets a **minus sign**: it opposes fast motion. It's a brake that grows
stronger the faster you're moving.


## 2. Build the PD controller and watch it damp overshoot

We take the derivative of the **measured altitude** (rate of climb) and subtract a share of it.
Compare P-only (overshoots and rings) with PD (glides in and stops).


In [ ]:
class PController:
    def __init__(self, Kp): self.Kp = Kp
    def __call__(self, target, measured, dt): return self.Kp * (target - measured)

class PDController:
    """Proportional + Derivative. D looks at how fast altitude is changing and brakes."""
    def __init__(self, Kp, Kd):
        self.Kp, self.Kd = Kp, Kd
        self.prev_measured = None
    def __call__(self, target, measured, dt):
        error = target - measured
        if self.prev_measured is None:
            rate = 0.0
        else:
            rate = (measured - self.prev_measured) / dt    # how fast altitude is changing
        self.prev_measured = measured
        # P pushes toward target; D pushes AGAINST fast motion (the minus sign = damping/brake)
        return self.Kp * error - self.Kd * rate

res_P  = simulate(PController(Kp=8.0),          total_time=10.0)
res_PD = simulate(PDController(Kp=8.0, Kd=4.0), total_time=10.0)

plt.figure(figsize=(9, 5))
plt.plot(res_P["t"],  res_P["x"],  color="tab:orange", lw=2, label="P only (overshoots, rings)")
plt.plot(res_PD["t"], res_PD["x"], color="tab:blue",   lw=2, label="PD (damped, glides in)")
plt.axhline(10, color="seagreen", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Derivative control acts like a shock absorber")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("Same Kp for both. Adding D removes most of the overshoot and settles the drone faster.")
print("(Both still sit a touch below 10 -- that's the steady-state error; only I fixes that.)")


## 3. 🎬 D calming an oscillating drone

Left idea vs right idea, in motion: without D the drone bobs past the line and rings; with D it eases
in. Run it and compare the two behaviours.


In [ ]:
res = simulate(PDController(Kp=10.0, Kd=5.0), total_time=8.0)
t, x = res["t"], res["x"]
fids = np.linspace(0, len(t)-1, 110).astype(int)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1); ax.set_ylim(0, 13); ax.set_xticks([])
ax.set_ylabel("altitude (m)"); ax.set_title("PD control (well damped)")
ax.axhline(0, color="saddlebrown", lw=3); ax.axhline(10, color="seagreen", ls="--", lw=2)
drone, = ax.plot([], [], "o", color="tab:blue", markersize=26)
label = ax.text(-0.9, 12, "", fontsize=11)
def init(): drone.set_data([], []); label.set_text(""); return drone, label
def update(f):
    i = fids[f]; drone.set_data([0], [x[i]])
    label.set_text(f"t={t[i]:.1f}s\nx={x[i]:.2f} m"); return drone, label
ani = animation.FuncAnimation(fig, update, frames=len(fids), init_func=init, blit=True, interval=45)
plt.close(fig); HTML(ani.to_jshtml())


## 4. The dark side: D **amplifies sensor noise**

Real sensors are noisy — every reading jitters by a little random amount. The P and I terms mostly
average that jitter out. But D computes *(new − old) / dt*, and with a tiny `dt` (say 0.02 s), even
a small random wiggle gets **divided by a tiny number = multiplied hugely.** The result: the D term
goes berserk and the thrust command turns to violent jitter.

Let's turn on sensor noise (our engine has a `noise_std` switch) and watch the PD controller's
thrust explode.


In [ ]:
# same PD gains, but now with noisy altitude readings (noise_std = 0.05 m)
res_clean = simulate(PDController(Kp=8.0, Kd=4.0), total_time=8.0, noise_std=0.0)
res_noisy = simulate(PDController(Kp=8.0, Kd=4.0), total_time=8.0, noise_std=0.05, seed=1)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6))
a1.plot(res_clean["t"], res_clean["thrust"], color="tab:blue",  lw=1.5, label="clean sensor")
a1.plot(res_noisy["t"], res_noisy["thrust"], color="tab:red",   lw=0.8, alpha=0.8, label="noisy sensor")
a1.set_ylabel("thrust command (N)"); a1.set_title("Tiny sensor noise -> the D term makes the thrust SCREAM")
a1.legend(); a1.grid(True, alpha=0.3)
a2.plot(res_noisy["t"], res_noisy["x"], color="tab:red", lw=1)
a2.axhline(10, color="seagreen", ls="--"); a2.set_ylabel("altitude (m)"); a2.set_xlabel("time (s)")
a2.set_title("The drone still flies, but the motors are being hammered by noise"); a2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("The blue thrust is smooth; the red is a jittery mess -- and that mess would burn out real motors.")


### Why does this happen? (a 20-second demo)

Take a smooth line, add a *little* noise, then look at its slope (derivative). The signal barely
changes — but its **slope** is wildly spiky, because differentiating turns small fast wiggles into
big spikes. This is a mathematical fact about derivatives, not a bug in our code.


In [ ]:
tt = np.linspace(0, 2, 200)
smooth = tt.copy()                                  # a gently rising line
noisy  = smooth + np.random.default_rng(0).normal(0, 0.02, tt.size)   # tiny noise added
dsmooth = np.diff(smooth) / np.diff(tt)
dnoisy  = np.diff(noisy)  / np.diff(tt)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(tt, smooth, label="smooth"); a1.plot(tt, noisy, lw=0.8, label="+ tiny noise")
a1.set_title("The signals look almost identical"); a1.legend(); a1.grid(True, alpha=0.3)
a2.plot(tt[1:], dsmooth, label="slope of smooth")
a2.plot(tt[1:], dnoisy, lw=0.8, alpha=0.8, label="slope of noisy")
a2.set_title("...but their SLOPES are worlds apart"); a2.legend(); a2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Same-looking signals, but the derivative amplifies the noise into huge spikes. That's D's curse.")


## 5. Two standard fixes: filtering + derivative-on-measurement

**Fix 1 — Low-pass filter the derivative.** Instead of using the raw, spiky rate of change, we
*smooth* it: keep a running "smoothed slope" and nudge it a little toward each new raw value. A
small filter factor `alpha` (e.g. 0.1) means "trust mostly the smoothed history, only 10% the noisy
new sample." This kills the spikes while keeping the useful trend.

```
smoothed = smoothed + alpha * (raw_rate - smoothed)   # low-pass filter
```

**Fix 2 — Derivative on measurement (not error).** We already take the derivative of the *measured
altitude* rather than the *error*. This has a bonus: when you suddenly change the *target*, the
error jumps instantly, and differentiating that jump would cause a violent "derivative kick." Using
the measurement avoids that kick. (We've been doing this all along — now you know why.)

Let's add the filter and compare noisy-unfiltered vs noisy-filtered.


In [ ]:
class PDController_Filtered:
    def __init__(self, Kp, Kd, alpha=0.1):
        self.Kp, self.Kd, self.alpha = Kp, Kd, alpha
        self.prev_measured = None
        self.smoothed_rate = 0.0
    def __call__(self, target, measured, dt):
        error = target - measured
        raw = 0.0 if self.prev_measured is None else (measured - self.prev_measured) / dt
        self.prev_measured = measured
        self.smoothed_rate += self.alpha * (raw - self.smoothed_rate)   # low-pass filter
        return self.Kp * error - self.Kd * self.smoothed_rate

res_raw  = simulate(PDController(Kp=8.0, Kd=4.0),                 total_time=8.0, noise_std=0.05, seed=1)
res_filt = simulate(PDController_Filtered(Kp=8.0, Kd=4.0, alpha=0.1), total_time=8.0, noise_std=0.05, seed=1)

plt.figure(figsize=(9, 5))
plt.plot(res_raw["t"],  res_raw["thrust"],  color="tab:red",   lw=0.8, alpha=0.7, label="D unfiltered (screaming)")
plt.plot(res_filt["t"], res_filt["thrust"], color="tab:green", lw=1.5, label="D filtered (calm)")
plt.xlabel("time (s)"); plt.ylabel("thrust command (N)")
plt.title("Low-pass filtering the derivative tames the noise")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("Same noisy sensor, same gains -- filtering turns a motor-destroying command into a smooth one.")


## 6. 🎛️ Play: Kd, noise, and the filter

Turn `Kd` up to damp harder (too much makes it sluggish). Add `noise` and watch the thrust get
jittery, then increase the smoothing (**lower** `alpha`) to calm it. This slider builds the exact
intuition you need for tuning real, noisy systems.


In [ ]:
def explore_PD(Kp=8.0, Kd=4.0, noise=0.03, alpha=0.1):
    ctrl = PDController_Filtered(Kp, Kd, alpha=alpha)
    res = simulate(ctrl, total_time=8.0, noise_std=noise, seed=2)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    a1.plot(res["t"], res["x"], color="tab:blue", lw=1.5); a1.axhline(10, color="seagreen", ls="--")
    a1.set_ylabel("altitude (m)"); a1.set_ylim(0, 14); a1.grid(True, alpha=0.3)
    a1.set_title(f"Kp={Kp}, Kd={Kd}, noise={noise}, filter alpha={alpha}")
    a2.plot(res["t"], res["thrust"], color="tab:purple", lw=1)
    a2.set_ylabel("thrust (N)"); a2.set_xlabel("time (s)"); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

interact(explore_PD,
         Kp=FloatSlider(min=2.0, max=16.0, step=1.0, value=8.0),
         Kd=FloatSlider(min=0.0, max=12.0, step=0.5, value=4.0),
         noise=FloatSlider(min=0.0, max=0.1, step=0.01, value=0.03),
         alpha=FloatSlider(min=0.02, max=1.0, step=0.02, value=0.1));


## ✅ Summary
- **Derivative control** uses the **rate of change** of the measurement: `D contribution = −Kd × (rate)`.
- It **damps** overshoot and oscillation — a shock absorber that opposes fast motion.
- D **amplifies sensor noise**, because differentiating turns small fast wiggles into big spikes.
- Fix it with a **low-pass filter** on the derivative and by taking the derivative **on the
  measurement** (which also avoids derivative kick on setpoint changes).


## ⚠️ Common mistakes
- **Cranking Kd to kill all overshoot.** Too much D makes the drone sluggish and *amplifies noise*.
- **Taking the derivative of the error with a jumpy setpoint.** Causes a violent derivative kick —
  differentiate the measurement instead.
- **Ignoring noise until hardware.** In sim it's tempting to leave noise off; then D looks perfect
  and betrays you on a real drone. Always test D with noise on.

## 🧭 Mental model
> *"P is a spring, I is a patient ledger-keeper, and D is a shock absorber. D doesn't care where you
> are — only how fast you're moving — and it pushes back on speed."*

## 🌍 Real-world applications
Car suspension, camera gimbal stabilization, elevator ride comfort, and the damping term in every
drone's attitude controller.


## 🧪 Exercises

**L1 — Observe.** In Section 2, remove D (set `Kd=0`) in the PD run. What comes back?
<details><summary>▸ Solution</summary>
The **overshoot and ringing** return — you're back to a pure P response. D was what damped them.
</details>

**L2 — Modify.** In the noisy comparison, set the filter `alpha=1.0` (no smoothing). What happens
to the "filtered" thrust?
<details><summary>▸ Solution</summary>
With <code>alpha=1.0</code> the filter passes the raw derivative unchanged, so the thrust becomes
just as jittery as the unfiltered case. Smoothing only helps when <code>alpha</code> is small.
</details>

**L3 — Predict.** You double `Kd` on a *noisy* sensor. Better tracking or worse jitter? Predict,
then test with the slider.
<details><summary>▸ Solution</summary>
**Worse jitter.** Doubling Kd doubles how strongly the (noisy) derivative drives the motors. More
damping, but a much noisier command — you'd need more filtering to compensate.
</details>


## ❓ Quick quiz
**Q1.** The derivative term responds to…
<details><summary>▸ Answer</summary>The **rate of change** (slope) of the signal — how fast it's moving, not where it is.</details>

**Q2.** Why does D amplify noise?
<details><summary>▸ Answer</summary>Differentiating divides small fast wiggles by a tiny dt, turning them into large spikes.</details>

**Q3.** A low-pass filter on D does what?
<details><summary>▸ Answer</summary>**Smooths** the derivative, removing the noise spikes while keeping the useful trend.</details>

**Q4.** Taking the derivative on the measurement instead of the error avoids…
<details><summary>▸ Answer</summary>**Derivative kick** — a violent spike when the target (setpoint) suddenly changes.</details>


## 🔭 Preview of Notebook 07 — *The Complete PID Controller*

You now have all three pieces: **P** (present), **I** (past), **D** (future). In Notebook 07 we
combine them into one clean, reusable `PIDController` class — with anti-windup and derivative
filtering built in — and drive a **live dashboard** showing altitude, velocity, thrust, and the
individual P/I/D contributions. This is where it all clicks together. 🚁
